In [ ]:
import mne
mne.viz.set_3d_backend('pyvistaqt')
import numpy as np
import matplotlib.pyplot as plt

# load data
filepath = "/Users/archinarang/Desktop/NeuroProject/Preprocessed/sub-sd021-epo.fif"
epochs = mne.read_epochs(filepath, preload=True)

epochs.set_eeg_reference('average', projection=True).apply_proj()



# S14: Standard Own 
# S15: Deviant Own 
sn_std = epochs['S14'].average()
sn_dev = epochs['S15'].average()

# N1 peak alignment
n1_window = sn_std.copy().pick(['Cz']).crop(tmin=0.05, tmax=0.15)
n1_idx = np.argmin(n1_window.data[0, :]) 
n1_time = n1_window.times[n1_idx]        

print(f"Subject 21's N1 Peak found at: {n1_time * 1000:.1f} ms")

# shifting for N1 peak alignment
sn_std.shift_time(-n1_time, relative=True)
sn_dev.shift_time(-n1_time, relative=True)


# MMN = [Deviant] - [Standard]
mmn_evoked = mne.combine_evoked([sn_dev, sn_std], weights=[1, -1])

# plotting the resulting MMN to verify
fig1 = mmn_evoked.plot_joint(times=[0.02, 0.05, 0.07], 
                             title="Sub-21: N1-Aligned MMN (Deviant Own - Standard Own)")


# calculating background noise using the pre-stimulus baseline of the standard trials
print("Computing Noise Covariance Matrix...")
noise_cov = mne.compute_covariance(epochs['S14'], tmin=-0.3, tmax=0.0, 
                                   method='auto', rank=None, verbose=True)

# covariance matrix 
fig2 = mne.viz.plot_cov(noise_cov, epochs.info, show_svd=False)

plt.show()

In [ ]:
import os.path as op
from mne.minimum_norm import make_inverse_operator, apply_inverse


# making virtual head 
print("\nFetching fsaverage template brain...")
fs_dir = mne.datasets.fetch_fsaverage(verbose=False)
subjects_dir = op.dirname(fs_dir)

# 10-20 layout
mmn_evoked.set_montage('standard_1020', match_case=False, on_missing='ignore')

# important 
trans = 'fsaverage'  
src = op.join(fs_dir, 'bem', 'fsaverage-ico-5-src.fif')
bem = op.join(fs_dir, 'bem', 'fsaverage-5120-5120-5120-bem-sol.fif')

print("Calculating Forward Solution (Leadfield Matrix)...")
fwd = mne.make_forward_solution(mmn_evoked.info, trans=trans, src=src,
                                bem=bem, eeg=True, meg=False, verbose=False)

# inverse operator 
print("Building Inverse Operator...")

inv = make_inverse_operator(mmn_evoked.info, fwd, noise_cov, 
                            loose=0.2, depth=0.8, verbose=False)

# applying sLORETA 
snr = 3.0
lambda2 = 1.0 / snr ** 2
print("Applying sLORETA to the MMN...")
stc = apply_inverse(mmn_evoked, inv, lambda2, method='sLORETA', verbose=False)

# plotting
brain = stc.plot(subjects_dir=subjects_dir, subject='fsaverage', surface='inflated', 
                 hemi='both', views=['lat', 'med'], initial_time=0.050, 
                 time_viewer=True, size=(800, 400))